# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Chosen Lane: Lane 2 — Refresh / Content Opportunity Scoring

**ML Task Type:** Binary Classification & Ranking/Scoring System

**Why this task type?**
1. **Binary Classification ($y \in \{0, 1\}$):** We need to identify whether a specific piece of content is undergoing severe decay and requires an editorial refresh (`1`) or if it is maintaining stable performance (`0`).
2. **Probability Scoring & Ranking:** Content teams operate with limited human bandwidth. By predicting continuous probability scores $P(y=1)$, we can rank all content assets into an actionable queue. Editors then prioritize the top-$K$ highest risk assets rather than sifting through thousands of pages manually.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target Proxy Definition: `target_needs_refresh`

**What we are predicting:**
A binary composite proxy flag (`1` or `0`) indicating an urgent content refresh candidate.

**How the proxy is constructed:**
Since real-world content decay isn't labeled by hand, we define a target rule based on observable behavior in the data:
- `1` (Needs Refresh): A page with established search demand (`impressions_90d >= 500`) that is actively decaying (`trend_direction == 'down'`) and mature enough to evaluate (`content_age_days >= 90`).
- `0` (Healthy / Leave Alone): Pages that are growing/stable (`trend_direction in ['up', 'stable']`) or low-demand pages where rewriting isn't cost-effective.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success Metric: **Precision@K**

**Why Precision@K?**
Content editors have a fixed operational capacity (e.g., $K = 50$ or $100$ pages per week).
- **Precision@K** measures what percentage of our top $K$ recommended pages *actually* needed an update.
- A false positive (flagging a healthy page) wastes expensive editorial writing time. Maximizing Precision@K ensures we don't send human writers down rabbit holes.

**Secondary Evaluation Metrics:**
- **PR-AUC (Precision-Recall Area Under Curve):** Better suited than ROC-AUC for imbalanced datasets where declining pages represent a subset of total inventory.
- **Recall:** Monitors how many total decaying assets we catch, ensuring revenue-driving pages don't quietly slip to zero traffic.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Unit of Analysis: 1 Row = 1 Unique Content Asset (`content_id`)

The unit of analysis represents a unique published content asset evaluated across aggregate performance windows (30-day, 90-day performance trends, word count, CTR, and scroll depth metrics).

In [9]:
import pandas as pd
import numpy as np

# Set Colab File Path
PATH = '/content/content_refresh_anonymized.csv'

df = pd.read_csv(PATH)

df_cleaned = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates(subset=['content_id']).copy()

df_cleaned['target_needs_refresh'] = (
    (df_cleaned['impressions_90d'] >= 500) &
    (df_cleaned['trend_direction'] == 'down')
).astype(int)

selected_cols = [
    'content_id', 'client_id', 'content_age_days',
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'trend_direction', 'trend_pct', 'target_needs_refresh'
]

print(f"Total Unit of Analysis Rows (Unique Pages): {len(df_cleaned):,}")
df_cleaned[selected_cols].head(10)

Total Unit of Analysis Rows (Unique Pages): 30,000


,content_id,client_id,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct,target_needs_refresh
0,content_304f48230142,client_f369cb89fc,187,3803,29,0.76,10.6,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,445,15320,7,0.05,20.3,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,141,12581,11,0.09,36.5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,463,11751,58,0.49,6.2,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,19140,24,0.13,44.0,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,147,3970,1,0.03,8.5,down,-38.9,1
6,content_9a34b442b552,client_8722616204,90,20,0,0.00,7.0,down,-92.3,0
7,content_a63219c6e95a,client_19581e27de,445,1724,1,0.06,21.2,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,90,32574,29,0.09,46.0,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,257,1240,2,0.16,4.9,down,-29.2,1


## 5. Why Machine Learning Beats Static Rules

1. **Complex Multi-Variable Interactions:** Simple SQL/if-statements (e.g., `impressions_drop > 20%`) miss complex non-linear signals. A page might have stable impressions but declining scroll depth (`scroll_rate`), dropping CTR, or position drift (`ctravg_position`). ML models synthesize these high-dimensional signals simultaneously.
2. **Continuous Prioritization vs. Rigid Binary Cuts:** Hardcoded rules group pages into arbitrary buckets (Yes/No). ML provides a calibrated continuous risk score ($0.0 \to 1.0$), letting teams dynamically adjust thresholds based on weekly resource availability.
3. **Adaptability Across Client Domains:** Different clients (`client_id`) have vastly different baselines for traffic, word counts, and search intent. Machine learning learns generalizable decay signatures without needing manual rule adjustments for every new client portfolio.

In [10]:
# Multi-variable feature summary comparing healthy (0) vs. refresh target (1)
summary_stats = df_cleaned.groupby('target_needs_refresh')[[
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'scroll_rate', 'engagement_rate', 'trend_pct'
]].mean()

print("--- Feature Average Comparison: Healthy (0) vs. Refresh Target (1) ---")
summary_stats.T

--- Feature Average Comparison: Healthy (0) vs. Refresh Target (1) ---


target_needs_refresh,0,1
impressions_90d,3840.943360,7935.179701
clicks_90d,13.364938,21.594217
ctr,0.647869,0.234851
avg_position,16.882868,15.255055
scroll_rate,21.308479,12.020562
engagement_rate,2.349779,2.906171
trend_pct,22.999429,-51.232577


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [11]:
# Final validation script
print("================ SECTOR SELF-CHECK ================")
print(f"✓ Task Type Defined: Binary Classification & Risk Scoring")
print(f"✓ Target Proxy Column Created: 'target_needs_refresh'")
print(f"✓ Unit of Analysis Verified: 1 Row = 1 Unique Content Asset")
print(f"✓ Total Evaluated Rows: {len(df_cleaned):,}")
print(f"✓ Class Distribution:")
print(df_cleaned['target_needs_refresh'].value_counts(normalize=True).map('{:.2%}'.format))
print("✓ No PII, URLs, or unanonymized client metrics exposed.")
print("===================================================")

================ SECTOR SELF-CHECK ================
✓ Task Type Defined: Binary Classification & Risk Scoring
✓ Target Proxy Column Created: 'target_needs_refresh'
✓ Unit of Analysis Verified: 1 Row = 1 Unique Content Asset
✓ Total Evaluated Rows: 30,000
✓ Class Distribution:
target_needs_refresh
0    66.80%
1    33.20%
Name: proportion, dtype: object
✓ No PII, URLs, or unanonymized client metrics exposed.
